# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:

task_type = "classification"
why = (
    "This is classification because the output is a binary decision: decline risk vs stable/improving. "
    "That directly supports a refresh-priority decision."
)
print(task_type)
print(why)

classification
This is classification because the output is a binary decision: decline risk vs stable/improving. That directly supports a refresh-priority decision.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
target = "is_declining_label"
label_source = (
    "The label comes from the downstream traffic trend direction in the data: "
    "trend_direction == 'down'. This is an observed outcome for the page."
)
print(target)
print(label_source)

is_declining_label
The label comes from the downstream traffic trend direction in the data: trend_direction == 'down'. This is an observed outcome for the page.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
success_metric = "precision@K for the top-ranked high-risk pages"
good_threshold = "Better than a simple rule-based baseline using only position or impressions."
print(success_metric)
print(good_threshold)

precision@K for the top-ranked high-risk pages
Better than a simple rule-based baseline using only position or impressions.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
import pandas as pd
from pathlib import Path

candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("..", "data", "raw", "content_refresh_anonymized.csv"),
    Path("..", "..", "data", "raw", "content_refresh_anonymized.csv"),
]
file_path = None
for candidate in candidates:
    if candidate.exists():
        file_path = candidate.resolve()
        break
if file_path is None:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv from the notebook working directory. "
        f"Tried: {[str(p) for p in candidates]}"
    )

print(f"Loading data from: {file_path}")

df = pd.read_csv(file_path)

unit_of_analysis = "one row = one pseudonymized content item (page)"
print(unit_of_analysis)
print(f"Dataset shape: {df.shape}")
print(df[["content_id", "client_id", "trend_direction", "impressions_90d", "ctr", "avg_position", "content_type"]].head(5))

Loading data from: /home/otto/Documents/projects/flyrank-internship/data/raw/content_refresh_anonymized.csv
one row = one pseudonymized content item (page)
Dataset shape: (30000, 44)
             content_id          client_id trend_direction  impressions_90d  \
0  content_304f48230142  client_f369cb89fc            down             3803   
1  content_a1fb4e703a9e  client_4e07408562            down            15320   
2  content_9aa793d4d895  client_7f2253d7e2            down            12581   
3  content_331d6c4de07b  client_19581e27de          stable            11751   
4  content_d99b7a2d90ca  client_3fdba35f04            down            19140   

    ctr  avg_position     content_type  
0  0.76          10.6  keyword article  
1  0.05          20.3  keyword article  
2  0.09          36.5  keyword article  
3  0.49           6.2  keyword article  
4  0.13          44.0  keyword article  


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [9]:
import pandas as pd
import numpy as np

if "df" not in globals():
    raise RuntimeError("The dataframe 'df' is not defined. Run the previous cell to load the starter data first.")

print("Example rule: predict decline risk when average position is worse than page 1.")
print("This is too simple because decline risk also depends on impressions, freshness, engagement, and keyword context.")

position_table = (
    df
    .assign(position_bucket=lambda x: pd.cut(x["avg_position"].replace(0, np.nan), bins=[0, 3, 10, 20, 50, 100], right=False))
    .groupby("position_bucket")
    .agg(
        total=("content_id", "size"),
        declining=("trend_direction", lambda s: (s.str.lower() == "down").sum())
    )
    .reset_index()
)
position_table["decline_rate"] = position_table["declining"] / position_table["total"]
print(position_table)

Example rule: predict decline risk when average position is worse than page 1.
This is too simple because decline risk also depends on impressions, freshness, engagement, and keyword context.
  position_bucket  total  declining  decline_rate
0          [0, 3)    990        509      0.514141
1         [3, 10)  11843       6728      0.568099
2        [10, 20)   7366       4474      0.607385
3        [20, 50)   7271       4088      0.562234
4       [50, 100)   1310        454      0.346565


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.